<a href="https://colab.research.google.com/github/OpenTopography/STAC-Examples/blob/main/OT_STAC_Interactive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OT Raster STAC Interactive Query Example

This notebook lets users select a bounding box/spatial area of interest on an interactive map, query the OpenTopography Raster STAC Catalog for intersecting datasets in Cloud Optimized GeoTIFF (COG) format, choose a dataset to download, and generate visualizations from the selected data.

Notebook Workflow:

| Step | Purpose |
|------|---------|
| 1 | Install & import dependencies |
| 2 | Configuration |
| 3 | Connect to the OT STAC catalog |
| 4 | Define helper functions |
| 5 | Interactive map — Draw bounding box |
| 6 | Search catalog & select a dataset |
| 7 | View dataset metadata |
| 8 | Generate Color Hillshade |
| 9 | Download output files |
|||

Viswanath Nandigam, Matt Beckley

info@opentopography.org

Work is part of the OpenTopography project supported by NSF Award Numbers 2410799, 2410800 & 2410801


### Step 1 - Install and import dependencies

In [ ]:
import subprocess, sys

pkgs = [
    "pystac",         # STAC metadata handling
    "pystac-client",  # STAC API client
    "rasterio",       # Raster data I/O
    "ipyleaflet",     # Interactive maps in Jupyter
    "ipywidgets",     # UI widgets for interactivity
    "matplotlib",     # Plotting and visualization
    "Pillow"          # Image processing
    ]

print("Installing packages …")

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs)

import json, io, base64
from pathlib import Path
from datetime import datetime

import numpy as np
import rasterio
import pystac

import matplotlib.pyplot as plt
from PIL import Image

import ipywidgets as widgets
from ipyleaflet import (
    Map, DrawControl, GeoJSON, ImageOverlay,
    basemaps, LayersControl, ScaleControl
)
from IPython.display import display, HTML, clear_output

# Google Colab needs its custom widget manager explicitly enabled for
# ipywidgets interactivity (e.g. this notebook's dataset dropdown) to work —
# without it, widgets can render visually but never send interaction events
# back to the kernel. No-op outside Colab.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

print("Done. All dependencies ready.")

### Step 2 - Configuration

In [ ]:
# URL for the OpenTopography raster STAC catalog.
STAC_URL      = "https://portal.opentopography.org/stac/raster_catalog.json"

# Map defaults

# The drawing toolbar is only enabled when the map is zoomed in to this
# level or deeper. This helps avoid very large selections when resources are limited.
MIN_DRAW_ZOOM = 12

# Initial map view shown when the notebook starts. Coordinates are in (latitude, longitude) order.
DEFAULT_CENTER = (37.74, -119.56)  # Yosemite Valley
DEFAULT_ZOOM   = 12
TERRAIN_CMAP   = "terrain"         # matplotlib colormap for color hillshade

# Maximum number of intersecting datasets shown in dropdown
MAX_RESULTS   = 20

# Output directory for generated files
OUTPUT_DIR    = Path("ot_stac_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Shared state used across notebook cells
state = {
    "drawn_bbox"       : None,   # Bounding box from the drawn geometry: [west, south, east, north]
    "search_hits"      : [],     # Intersecting search results as (pystac.Collection, pystac.Item)
    "selected_col"     : None,   # Collection selected by the user
    "selected_item"    : None,   # Item selected by the user
    "hillshade_path"   : None,   # Path to the saved hillshade PNG
    "metadata_path"    : None,   # Path to the saved metadata JSON
    "overlay_layer"    : None,   # Active ImageOverlay currently shown on the map
    "bbox_layer"       : None,   # GeoJSON layer representing the drawn area
    "draw_ctrl_on_map" : False,  # Whether DrawControl is currently on the map
    "current_zoom"     : DEFAULT_ZOOM, # Current map zoom level
}

print(f"Outputs → {OUTPUT_DIR.resolve()}")
print(f"MIN_DRAW_ZOOM = {MIN_DRAW_ZOOM}  (zoom in to enable drawing)")

### Step 3 - Connect to the OT Raster STAC Catalog

In [ ]:
#Open the OT Raster STAC catalog (static)
print(f"Connecting to STAC catalog …\n  {STAC_URL}")
catalog = pystac.Catalog.from_file(STAC_URL)
print(f"Done. Connected to STAC Catalog")

# Iterate through the catalog's immediate children and keep only STAC
# collections. This creates a simple in-memory cache for later searches
# and dropdown population in the notebook.
print("Loading collections (walking static JSON links) …")
_collections_cache = [
    child for child in catalog.get_children()
    if isinstance(child, pystac.Collection)
]
print(f"Done. {len(_collections_cache)} collections loaded.")

### Step 4 - Helper Functions

In [ ]:
# ------------------------------------------------------------------------
# Geometry helpers
# ------------------------------------------------------------------------

def bbox_intersects(a, b):
    #Return true if [minx,miny,maxx,maxy] bboxes a and b overlap.
    return not (a[2] <= b[0] or a[0] >= b[2] or a[3] <= b[1] or a[1] >= b[3])

def geojson_rect_to_bbox(geo):
    #Convert a drawn rectangle GeoJSON feature to [west, south, east, north].
    coords = geo["geometry"]["coordinates"][0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

# ------------------------------------------------------------------------
# STAC asset detection
# ------------------------------------------------------------------------

#Collect all media-type strings from a STAC asset, checking standard attributes and extra_fields.
def _asset_type_strings(asset):
    candidates = []
    # Standard pystac Asset attribute
    candidates.append(getattr(asset, "media_type", "") or "")
    # extra_fields dict (used by some STAC flavours)
    ef = getattr(asset, "extra_fields", {}) or {}
    candidates.append(ef.get("type", "") or "")
    candidates.append(ef.get("media_type", "") or "")
    # The asset dict itself may have a top-level 'type' key
    if hasattr(asset, "to_dict"):
        d = asset.to_dict()
        candidates.append(d.get("type", "") or "")
    return [s.lower() for s in candidates if s]

def is_tiff_asset(asset):
    """
    Return True if an asset is strictly a GeoTIFF/Cloud Optimised GeoTIFF.
    Excludes directory-based legacy formats like Esri Binary Grids (.adf).
    """
    href = (getattr(asset, "href", "") or "").lower()

    # Explicitly skip legacy Esri Binary Grid files
    if href.endswith(".adf") or "/_adf_" in href:
        return False

    for s in _asset_type_strings(asset):
        if "tif" in s:
            return True

    if href.endswith((".tif", ".tiff")):
        return True

    # Cloud-optimised GeoTIFFs often have no extension — also accept any
    # asset whose role or key name suggests raster data
    ef = getattr(asset, "extra_fields", {}) or {}
    roles = ef.get("roles", []) or []
    if any(r in ("data", "overview", "visual") for r in roles):
        # Double check it isn't an adf path hidden without extension
        if ".adf" not in href:
            return True

    return False

def get_asset_bbox(asset, item):
    #Return asset's own bbox if present, otherwise fall back to item bbox.
    ef = getattr(asset, "extra_fields", {}) or {}
    return ef.get("bbox") or item.bbox

def find_tiff_assets(item, query_bbox):
    """
    Return every TIFF asset in the item whose own bbox intersects
    query_bbox. An AOI spanning a tile boundary needs every 
    intersecting tile, not just the first one found, to fully cover 
    the AOI. 
    """
    matches = []
    for a in item.assets.values():
        try:
            if is_tiff_asset(a) and bbox_intersects(get_asset_bbox(a, item), query_bbox):
                matches.append(a)
        except Exception:
            continue
    return matches

# ------------------------------------------------------------------------
# STAC search
# ------------------------------------------------------------------------

from concurrent.futures import ThreadPoolExecutor, as_completed

_items_cache = {}  # col.id → [pystac.Item, …]  populated on first fetch

def _item_has_intersecting_asset(item, bbox):
    #True if any of the item's own TIFF assets has a bbox that genuinely intersects.
    try:
        return any(
            is_tiff_asset(a) and bbox_intersects(get_asset_bbox(a, item), bbox)
            for a in item.assets.values()
        )
    except Exception:
        return False

def _check_collection(col, bbox):
    """
    Fetch all items for one collection (cached after first call) and return
    those with at least one asset whose own bbox genuinely intersects the
    query bbox. Checking per-asset bboxes instead of just the item's overall
    bbox matters for datasets with several disjoint coverage areas.
    """
    if col.id not in _items_cache:
        try:
            _items_cache[col.id] = list(col.get_items())
        except Exception:
            _items_cache[col.id] = []
    results = []
    for item in _items_cache[col.id]:
        try:
            # Cheap item-level pre-filter before scanning any per-asset bboxes.
            if item.bbox and bbox_intersects(item.bbox, bbox) and _item_has_intersecting_asset(item, bbox):
                results.append((col, item))
        except Exception:
            continue
    return results

# ------------------------------------------------------------------------
# Raster helpers
# ------------------------------------------------------------------------

MAX_DISPLAY_PX = 1024   # memory safeguard: array in RAM is never bigger than this
MIN_DISPLAY_PX = 512    # display floor: avoids a handful of blocky pixels on tiny AOIs

def read_raster_safe(hrefs, bbox=None, max_px=MAX_DISPLAY_PX, min_px=MIN_DISPLAY_PX):
    """
    Open one or more COG tiles and return a single mosaicked preview band
    covering the area of interest, without loading full-resolution data
    into RAM. 

    If `bbox` ([west, south, east, north] in EPSG:4326) is given, only that
    window is read. If omitted, the union of all tiles' extents is used.
    """
    from contextlib import ExitStack
    from rasterio.enums import Resampling
    from rasterio.merge import merge as rio_merge
    from rasterio.warp import transform_bounds
    from rasterio.coords import BoundingBox

    with ExitStack() as stack:
        datasets = [stack.enter_context(rasterio.open(href)) for href in hrefs]
        crs = datasets[0].crs
        nodata = datasets[0].nodata

        if bbox is not None:
            west, south, east, north = transform_bounds("EPSG:4326", crs, *bbox)
        else:
            lefts, bottoms, rights, tops = zip(*(ds.bounds for ds in datasets))
            west, south, east, north = min(lefts), min(bottoms), max(rights), max(tops)

        # Estimate the native pixel size from the first tile, to fit the
        # merged AOI's longer side between min_px and max_px — the same
        # memory safeguard / display floor used for a single-tile read.
        px, py = abs(datasets[0].transform.a), abs(datasets[0].transform.e)
        src_w = max(1, (east - west) / px)
        src_h = max(1, (north - south) / py)
        longest = max(src_w, src_h)
        scale = min(max_px, max(min_px, longest)) / longest
        out_w = max(2, round(src_w * scale))
        out_h = max(2, round(src_h * scale))

        # Always pass 0 as merge()'s own nodata argument rather than the
        # dataset's real value. rasterio.merge has a bug/limitation. 0 
        # always passes that safety check, so merge() just copies raw 
        # pixel values through; the REAL nodata value still correctly 
        # mask nodata afterward, using ds.nodata rather than 0.
        mosaic, _ = rio_merge(
            datasets,
            bounds=(west, south, east, north),
            res=((east - west) / out_w, (north - south) / out_h),
            resampling=Resampling.bilinear,
            nodata=0,
        )

    band = mosaic[0].astype("float32")
    if nodata is not None:
        band[band == nodata] = np.nan
    band[band < -1e30] = np.nan  # large float sentinels in some Arc/ASCII-derived grids
    band[band > 1e30] = np.nan

    # Pixel spacing in real-world units (native CRS), needed for correct
    # hillshade slope/aspect math.
    if crs.is_geographic:
        mid_lat = (south + north) / 2.0
        dx = ((east - west) / out_w) * 111_320.0 * np.cos(np.deg2rad(mid_lat))
        dy = ((north - south) / out_h) * 111_132.0
    else:
        dx = (east - west) / out_w
        dy = (north - south) / out_h

    # `bounds` is returned in EPSG:4326 (lat/lon) regardless of the
    # dataset's native CRS. Callers use it to center/overlay the preview on
    # an ipyleaflet map, which always expects geographic coordinates — for
    # a projected dataset, west/south/east/north above are in meters, 
    # not degrees, and handing those to the map directly sends it to an 
    # improper location 
    disp_west, disp_south, disp_east, disp_north = transform_bounds(crs, "EPSG:4326", west, south, east, north)
    bounds = BoundingBox(disp_west, disp_south, disp_east, disp_north)

    return band, dx, dy, bounds, crs, band.shape

# ------------------------------------------------------------------------
# Hillshade & colour rendering
# ------------------------------------------------------------------------

def compute_hillshade(arr, azimuth=315.0, altitude=45.0, dx=1.0, dy=1.0):
    """
    Lambertian hillshade from a 2-D elevation array.
    azimuth is the sun's compass bearing (degrees); altitude is its angle
    above the horizon (degrees). Returns a uint8 array (0-255).
    """
    nan_mask = np.isnan(arr)
    if nan_mask.any():
        fill = np.nanmean(arr)
        arr = np.where(nan_mask, 0.0 if np.isnan(fill) else fill, arr)

    gy = np.gradient(arr, dy, axis=0)
    gx = np.gradient(arr, dx, axis=1)
    slope  = np.pi / 2.0 - np.arctan(np.hypot(gx, gy))
    aspect = np.arctan2(-gx, gy)
    az, alt = np.deg2rad(azimuth), np.deg2rad(altitude)
    hs = np.sin(alt)*np.sin(slope) + np.cos(alt)*np.cos(slope)*np.cos(az - aspect)
    return (np.clip(hs, 0, 1) * 255).astype(np.uint8)


def make_color_hillshade_rgba(band, cmap_name=TERRAIN_CMAP,
                               azimuth=315.0, altitude=45.0,
                               dx=1.0, dy=1.0, blend=0.6):
    """
      Return a PIL RGBA image blending terrain colour with hillshade relief.
      blend=0 → gives pure greyscale hillshade
      blend=1 → gives pure terrain colour (no relief)
    """
    valid = ~np.isnan(band)
    if not valid.any():
        # Entire array is nodata — return a fully transparent image
        return Image.fromarray(np.zeros((*band.shape, 4), dtype=np.uint8))

    alpha = (valid * 255).astype(np.uint8)

    vmin, vmax = float(np.nanmin(band)), float(np.nanmax(band))
    norm = np.where(valid, (band - vmin) / max(vmax - vmin, 1e-9), 0.0)
    color_rgba = (plt.get_cmap(cmap_name)(norm) * 255).astype(np.uint8)

    hs      = compute_hillshade(band, azimuth=azimuth, altitude=altitude, dx=dx, dy=dy)
    hs_norm = hs.astype("float32") / 255.0

    rgb     = color_rgba[..., :3].astype("float32") / 255.0
    blended = np.clip(rgb * (blend + (1 - blend) * hs_norm[..., None]), 0, 1)
    return Image.fromarray(np.dstack([(blended * 255).astype(np.uint8), alpha]))


def pil_to_data_url(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


print("Done. Helper functions defined.")

### Step 5 - Interactive Map: Draw Bounding Box

In [ ]:
# ------------------------------------------------------------------------
# Map setup 
# ------------------------------------------------------------------------

if "m" not in globals():
    m = Map(
        center=DEFAULT_CENTER,
        zoom=DEFAULT_ZOOM,
        scroll_wheel_zoom=True,
        basemap=basemaps.Esri.WorldImagery,
    )
    m.layout.height = "540px"
    m.add_control(LayersControl(position="topright"))
    m.add_control(ScaleControl(position="bottomleft"))

    draw_ctrl = DrawControl(
        rectangle={"shapeOptions": {"color": "#f7b731", "weight": 3}},
        polyline={}, polygon={}, circle={}, marker={}, circlemarker={},
    )

    zoom_label = widgets.HTML()
    bbox_info = widgets.HTML()

    # Enable drawing only when the user is zoomed in far enough to make a reasonable selection.
    def on_zoom_change(change):
        z = change["new"]
        state["current_zoom"] = z
        if z >= MIN_DRAW_ZOOM:
            if not state["draw_ctrl_on_map"]:
                m.add_control(draw_ctrl)
                state["draw_ctrl_on_map"] = True
            zoom_label.value = (
                f"<span style='color:#27ae60;font-weight:bold'>"
                f"Drawing enabled (zoom {z} ≥ {MIN_DRAW_ZOOM})"
                f" — draw a rectangle on the map</span>"
            )
        else:
            if state["draw_ctrl_on_map"]:
                m.remove_control(draw_ctrl)
                state["draw_ctrl_on_map"] = False
            zoom_label.value = (
                f"<span style='color:#e67e22;font-weight:bold'>"
                f"Zoom in to level {MIN_DRAW_ZOOM}+ to enable drawing "
                f"(current: {z})</span>"
            )

    m.observe(on_zoom_change, names=["zoom"])

    # Save the rectangle drawn by the user, convert it to a bounding box, and
    # show the selected area on the map. The saved geometry remains available
    # even if the user later zooms back out.
    def on_draw(self, action, geo_json):
        if action != "created":
            return
        if geo_json.get("geometry", {}).get("type", "") not in ("Polygon", "Rectangle"):
            return

        bbox = geojson_rect_to_bbox(geo_json)
        state["drawn_bbox"] = bbox

        w, s, e, n = bbox
        bbox_info.value = (
            f"<b>Bounding Box:</b> "
            f"W={w:.5f}  S={s:.5f}  E={e:.5f}  N={n:.5f}"
        )

        # Replace previous bbox highlight layer
        if state["bbox_layer"] is not None:
            try: m.remove_layer(state["bbox_layer"])
            except Exception: pass

        rect_layer = GeoJSON(
            data={"type": "Feature",
                  "geometry": {"type": "Polygon",
                               "coordinates": [[[w,s],[e,s],[e,n],[w,n],[w,s]]]},
                  "properties": {}},
            style={"color": "#f7b731", "weight": 3, "fillOpacity": 0.08},
            name="Search Area"
        )
        m.add_layer(rect_layer)
        state["bbox_layer"] = rect_layer

        print(f"Done. Bounding box saved: {[round(v, 5) for v in bbox]}")
        print("Run step 6 to search the STAC catalog.")

    draw_ctrl.on_draw(on_draw)

# ------------------------------------------------------------------------
# Reset for this run - clears any previously drawn box and the previous
# hillshade overlay (both now stale if you're about to search a different
# region), without touching Step 6/7/8's own results. Removing the draw
# control unconditionally first (rather than trusting a flag) means this
# stays correct no matter what state it was left in.
# ------------------------------------------------------------------------

try: m.remove_control(draw_ctrl)
except Exception: pass

if state["bbox_layer"] is not None:
    try: m.remove_layer(state["bbox_layer"])
    except Exception: pass
if state["overlay_layer"] is not None:
    try: m.remove_layer(state["overlay_layer"])
    except Exception: pass

state["drawn_bbox"] = None
state["bbox_layer"] = None
state["overlay_layer"] = None
state["draw_ctrl_on_map"] = False
bbox_info.value = "<i>No bounding box drawn yet.</i>"

# Sync the drawing toolbar and its label to the map's current zoom, which
# may already be zoomed in from a previous run since the map is reused.
z = m.zoom
state["current_zoom"] = z
if z >= MIN_DRAW_ZOOM:
    m.add_control(draw_ctrl)
    state["draw_ctrl_on_map"] = True
    zoom_label.value = (
        f"<span style='color:#27ae60;font-weight:bold'>"
        f"Drawing enabled (zoom {z} ≥ {MIN_DRAW_ZOOM})"
        f" — draw a rectangle on the map</span>"
    )
else:
    zoom_label.value = (
        f"<span style='color:#e67e22;font-weight:bold'>"
        f"Zoom in to level {MIN_DRAW_ZOOM}+ to enable drawing "
        f"(current: {z})</span>"
    )

# ---------------------------------------------------------------------
# Render interface
# ---------------------------------------------------------------------

# Display the step title, status message, interactive map, and current
# bounding box summary as a single notebook layout.
display(widgets.VBox([
    widgets.HTML("<h4 style='margin:4px 0'> Draw your Spatial area of Interest</h4>"),
    zoom_label,
    m,
    bbox_info,
]))

### Step 6 - Search Catalog and Select Dataset

In [ ]:
# ---------------------------------------------------------------------
# Search STAC collections for datasets intersecting the drawn bbox
# ---------------------------------------------------------------------
if state["drawn_bbox"] is None:
    print("Draw a bounding box in Step 5 above first.")
else:
    bbox = state["drawn_bbox"]
    n = len(_collections_cache)
    print(f"Searching {n} collections")

    # Run parallel search with a live progress completed-count
    raw_hits = []
    completed = 0
    with ThreadPoolExecutor(max_workers=16) as pool:
        futures = {pool.submit(_check_collection, col, bbox): col for col in _collections_cache}
        for future in as_completed(futures):
            completed += 1
            raw_hits.extend(future.result())
            print(f"\r {completed}/{n} collections checked — {len(raw_hits)} raw hit(s) so far …", end="", flush=True)
            if len(raw_hits) >= MAX_RESULTS:
                for f in futures:
                    f.cancel()
                break

    print() # newline after progress tracking

    # Strictly filter hits to include collections that have a true COG/GeoTIFF asset
    hits = []
    for col, item in raw_hits:
        if any(is_tiff_asset(a) for a in item.assets.values()):
            hits.append((col, item))

    # Bound the results to MAX_RESULTS after the filter step
    hits = hits[:MAX_RESULTS]
    state["search_hits"] = hits

    if not hits:
        print("No native COG-formatted datasets intersect this bounding box.")
        print("Try drawing a larger bounding box over a data-rich area.")
    else:
        print(f"Done. Found {len(hits)} dataset(s) — select one below:")

        # Build the drop down dataset selector from the filtered results only
        opts = [
            (f"[{i}] {col.title or col.id} ▸ {item.id}", i)
            for i, (col, item) in enumerate(hits)
        ]

        dataset_dd = widgets.Dropdown(
            options=opts,
            value=0,
            description='Dataset:',
            layout=widgets.Layout(width='100%')
        )

        def on_dropdown_change(change):
            if change['type'] == 'change' and change['name'] == 'value':
                idx = change['new']
                selected_col, selected_item = state["search_hits"][idx]
                state["selected_col"] = selected_col
                state["selected_item"] = selected_item

        dataset_dd.observe(on_dropdown_change, names=["value"])

        # Default initial selection to the first valid filtered hit
        col0, item0 = hits[0]
        state["selected_col"] = col0
        state["selected_item"] = item0

        # Build the per-dataset details as an HTML widget rather than bare
        # print() calls. Colab doesn't reliably preserve execution order
        # between plain stdout output and rich display() output, so mixing
        # print() with a widget display can land the text above or below
        # the widget unpredictably. Putting the details in the SAME VBox as
        # the dropdown guarantees the dropdown renders first, since both
        # are part of one widget tree rendered in a single display() call.
        def _escape(s):
            return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

        detail_blocks = []
        for i, (col, item) in enumerate(hits):
            tiff_keys = [k for k, a in item.assets.items() if is_tiff_asset(a)]
            all_keys = list(item.assets.keys())
            detail_blocks.append(
                f" [{i:2d}] {_escape(col.title or col.id)}\n"
                f"      item id   : {_escape(item.id)}\n"
                f"      item bbox : {_escape([round(v,3) for v in item.bbox])}\n"
                f"      all assets: {_escape(all_keys)}\n"
                f"      COG keys  : {_escape(tiff_keys)}\n"
            )
        details_html = widgets.HTML(
            "<pre style='white-space:pre-wrap;font-size:12px;margin:0'>"
            + "\n".join(detail_blocks) +
            "</pre>"
        )

        # Collapse the details into an Accordion (collapsed by default).
        details_panel = widgets.Accordion(children=[details_html])
        details_panel.set_title(0, "Search details (click to expand)")
        details_panel.selected_index = None  # collapsed by default

        display(widgets.VBox([
            widgets.HTML("<h4 style='margin:6px 0 4px'>Select a Dataset</h4>"),
            dataset_dd,
            widgets.HTML("<i>Run Step 7 to load its full metadata.</i>"),
            details_panel,
        ]))

### Step 7 - Dataset Metadata

In [ ]:
# ---------------------------------------------------------------------
# Export selected dataset metadata to disk and show summary
# ---------------------------------------------------------------------
col  = state.get("selected_col")
item = state.get("selected_item")

if col is None or item is None:
    print("Run step 6 and select a dataset first.")
else:
    # Build metadata from the pystac objects already in memory
    meta_dict = col.to_dict()
    meta_dict["_selected_item"] = item.to_dict()

    ts        = datetime.now().strftime("%Y%m%d_%H%M%S")
    meta_path = OUTPUT_DIR / f"metadata_{col.id}_{ts}.json"
    meta_path.write_text(json.dumps(meta_dict, indent=2))
    state["metadata_path"] = meta_path

    # Print a compact, readable summary for quick inspection
    bar = "═" * 64
    print(bar)
    print(f"  COLLECTION  : {meta_dict.get('title', col.id)}")
    print(f"  ID          : {col.id}")
    desc = meta_dict.get("description", "")
    if desc:
        print(f"  DESCRIPTION : {desc[:200]}{'…' if len(desc)>200 else ''}")

    # Surface a few useful link types if present (about, license, citation)
    for lk in meta_dict.get("links", []):
        if lk.get("rel") in ("about", "license", "cite-as"):
            print(f"  {lk['rel'].upper():12}: {lk.get('href','')[:72]}")
    print(bar)
    print(f"  ITEM        : {item.id}")
    print(f"  ITEM BBOX   : {item.bbox}")
    print()

    print(f"\n Full metadata saved → {meta_path}")
    print("\n Run step 8 to generate the Color Hillshade.")

### Step 8 - Generate Color Hillshade

In [ ]:
item = state.get("selected_item")
col  = state.get("selected_col")
bbox = state.get("drawn_bbox")

if item is None or bbox is None:
    print("Complete steps 5–7 first.")
else:
    # Optional debugging block for asset inspection.
    # Uncomment if you need to diagnose which assets the item exposes.
    """
    print(f"Assets for item '{item.id}':")
    for k, a in item.assets.items():
        mt  = getattr(a, "media_type", "") or ""
        ef  = getattr(a, "extra_fields", {}) or {}
        t2  = ef.get("type", "") or ""
        detected = "✓ raster" if is_tiff_asset(a) else ""
        print(f"  [{k}]  media_type={mt!r}  ef.type={t2!r}  {detected}")
        print(f"    {a.href}")
    print()
    """
    # Gather every TIFF asset in this item whose own bbox intersects the
    # AOI - a global tile grid (COP30, NASADEM, SRTM, ALOS...) or a tiled
    # regional survey (like B4) needs more than one tile mosaicked
    # together to cover an AOI that straddles a tile boundary.
    assets = find_tiff_assets(item, query_bbox=bbox)

    if not assets:
        print("No usable raster asset found for this item.")
        print("Select a different dataset in step 6 and re-run step 7–8.")
    else:
        hrefs = [a.href for a in assets]
        print(f" {len(hrefs)} intersecting tile(s):")
        for href in hrefs:
            print(f"   {href}")

        print(f"\n Opening raster(s) (may take a moment for remote COG files) …")

        try:
            # Inspect the first tile's metadata for a quick sanity check.
            with rasterio.open(hrefs[0]) as ds:
                print(f"   CRS    : {ds.crs}")
                print(f"   Tile size (first tile): {ds.width} x {ds.height} px")
                print(f"   Bands  : {ds.count}")
                print(f"   Nodata : {ds.nodata}")
                overviews = ds.overviews(1)
                print(f"   COG overviews: {overviews if overviews else 'none'}")

            # Read + mosaic a preview clipped to the drawn AOI. rasterio's
            # merge() stitches the tiles together and resamples in one
            # step — the same idea a desktop GIS uses to combine adjacent
            # tiles — using only rasterio, no GDAL command-line tools.
            print(f"\n Reading + mosaicking AOI window at ≤{MAX_DISPLAY_PX}px")
            band, dx, dy, bounds, crs, shape = read_raster_safe(hrefs, bbox=bbox)
            print(f"   Read shape : {shape[1]} x {shape[0]} px (clipped to AOI)")
            print(f"   RAM usage  : ~{band.nbytes / 1e6:.1f} MB for this array")
            west, south, east, north = bounds.left, bounds.bottom, bounds.right, bounds.top

            if np.all(np.isnan(band)):
                print(" No data in the drawn AOI for this asset — try a larger box or a different dataset.")
            else:
                # Render a terrain-colored hillshade image from the preview array.
                print(f"\n Rendering color hillshade …")
                img = make_color_hillshade_rgba(band, dx=dx, dy=dy)

                # Save the rendered preview to disk.
                ts      = datetime.now().strftime("%Y%m%d_%H%M%S")
                hs_path = OUTPUT_DIR / f"hillshade_{col.id}_{ts}.png"
                img.save(str(hs_path))
                state["hillshade_path"] = hs_path

                # Show the rendered preview inline in matplotlib.
                fig, ax = plt.subplots(figsize=(10, 6))
                ax.imshow(img)
                ax.set_title(
                    f"{col.title or col.id}\n"
                    f"Elev range: {np.nanmin(band):.1f} – {np.nanmax(band):.1f} m  |  "
                    f"{img.size[0]}x{img.size[1]} px",
                    fontsize=10
                )
                ax.axis("off")
                plt.tight_layout()
                plt.show()

                # Replace any previous overlay and add the new image to the map (from step 5)
                if state["overlay_layer"] is not None:
                    try: m.remove_layer(state["overlay_layer"])
                    except Exception: pass

                overlay = ImageOverlay(
                    url=pil_to_data_url(img),
                    bounds=((south, west), (north, east)),
                    opacity=0.85,
                    name="Color Hillshade"
                )
                m.add_layer(overlay)
                m.center = ((south + north) / 2.0, (west + east) / 2.0)
                state["overlay_layer"] = overlay

                print(f"\n Done!")
                print(f"   Elevation range : {np.nanmin(band):.1f} – {np.nanmax(band):.1f} m")
                print(f"   Image size      : {img.size[0]}x{img.size[1]} px")
                print(f"   Saved           : {hs_path}")
                print("\n  Overlay added to the map — scroll up to step 5 to see it.")
                print("\n  Run step 9 to download all output files.")

        except Exception as ex:
            import traceback
            print(f"  Failed to open raster: {ex}")
            print(f"  Assets: {hrefs}")
            print("\n Full traceback:")
            traceback.print_exc()
            print("\nTip: try selecting a different dataset in step 6 and re-running steps 7–8.")

### Step 9 - Download Output Files

In [ ]:
# ---------------------------------------------------------------------
# List and download generated output files
# ---------------------------------------------------------------------
files = sorted(OUTPUT_DIR.iterdir())

if not files:
    print(" No output files yet — run steps 7 and 8 first.")
else:
    print(f" {len(files)} file(s) in {OUTPUT_DIR}:\n")
    for fp in files:
        # Read the file and build a browser-download link using a data URL.
        size_kb = fp.stat().st_size / 1024
        data    = fp.read_bytes()
        b64     = base64.b64encode(data).decode()
        mime    = "image/png" if fp.suffix == ".png" else "application/json"
        href    = f"data:{mime};base64,{b64}"

        display(HTML(
            f'<a href="{href}" download="{fp.name}" '
            f'style="display:inline-block;margin:4px 0;padding:7px 16px;'
            f'background:#2980b9;color:white;border-radius:5px;'
            f'text-decoration:none;font-size:13px;font-family:sans-serif;">'
            f' {fp.name} ({size_kb:.1f} KB)</a>'
        ))
    print()
    print(" In Colab you can also click the Files icon in the left sidebar,")
    print(" navigate to ot_stac_outputs/, right-click a file → Download.")